# Supporting notebook 4 · TF-IDF + logistic regression (member `tfidf`)

**Role in the solution.** The classic sparse baseline, used as the fourth level-1 member of the main notebook (stack
v10). It produced `outputs/oof_tfidf.npy` (47,817 × 3 out-of-fold probabilities) and `outputs/test_tfidf.npy`
(mean of the 5 fold models on test). OOF macro F1 0.5219.

| Setting | Value |
|---|---|
| Features | word 1–2-gram TF-IDF (min_df 2) + character 2–5-gram TF-IDF within word boundaries (min_df 3), sublinear tf; fitted on the training folds only |
| Model | `LogisticRegression(C=4, class_weight="balanced", max_iter=2000)` |
| Input | The raw post (the vectoriser lower-cases it) |
| Folds | the pinned `StratifiedKFold(5, shuffle=True, random_state=42)` over rows sorted by `id` |

**How to run.** Only the paths in section 0 need changing; the notebook trains all 5 folds on a CPU in a few minutes,
saves the fold models, and compares its predictions with the saved ones in `outputs/`.

**Exactness.** The saved `outputs/*_tfidf.npy` were made on 2026-09-27 by this code (the team's notebook 02) on another
machine and library build. Logistic regression's solver is not bit-stable across builds: re-running here gives OOF
0.5222 and 99.6% identical test predictions. The main notebook uses the saved files (the submitted ones).

## 0. Configuration — the only cell to edit

In [1]:
from pathlib import Path
DATA_DIR = Path("../../data")
OUT_DIR = Path("outputs")                                   # saved predictions used by the main notebook
RERUN_DIR = OUT_DIR / "rerun"                               # this notebook's own run
MODEL_DIR = Path("../../preprocessing_and_helper_files/fine_tuned_models/tfidf_logreg")

In [2]:
import numpy as np, pandas as pd, joblib, sklearn
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
mf1 = lambda t, p: f1_score(t, p, average="macro")
RERUN_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
tr = pd.read_csv(DATA_DIR / "train.csv").sort_values("id").reset_index(drop=True)
te = pd.read_csv(DATA_DIR / "test.csv").sort_values("id").reset_index(drop=True)
y = tr.y.values
folds = np.full(len(y), -1)
for i, (_, v) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(tr, y)):
    folds[v] = i
print("scikit-learn", sklearn.__version__, "| fold sizes", np.bincount(folds).tolist())

scikit-learn 1.9.1 | fold sizes [9564, 9564, 9563, 9563, 9563]


## 1. Train the 5 fold models (features fitted on the training folds only)

In [3]:
def feats(fit_texts, *others):
    w = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    X = hstack([w.fit_transform(fit_texts), c.fit_transform(fit_texts)]).tocsr()
    return (w, c), [X] + [hstack([w.transform(o), c.transform(o)]).tocsr() for o in others]


oof, test_p = np.zeros((len(tr), 3), np.float32), np.zeros((len(te), 3), np.float32)
for f in range(5):
    tri, vai = np.where(folds != f)[0], np.where(folds == f)[0]
    (w, c), (Xtr, Xva, Xte) = feats(tr.text.values[tri], tr.text.values[vai], te.text.values)
    m = LogisticRegression(C=4, class_weight="balanced", max_iter=2000).fit(Xtr, y[tri])
    oof[vai], test_p = m.predict_proba(Xva), test_p + m.predict_proba(Xte) / 5
    joblib.dump({"word_vectorizer": w, "char_vectorizer": c, "logreg": m, "fold": f}, MODEL_DIR / f"fold{f}.joblib", compress=3)
    print(f"fold {f}: macro F1 {mf1(y[vai], oof[vai].argmax(1)):.4f}  ({Xtr.shape[1]:,} features) -> {MODEL_DIR.name}/fold{f}.joblib")
np.save(RERUN_DIR / "oof_tfidf.npy", oof); np.save(RERUN_DIR / "test_tfidf.npy", test_p)
print(f"OOF macro F1 {mf1(y, oof.argmax(1)):.4f}")

fold 0: macro F1 0.5176  (160,185 features) -> tfidf_logreg/fold0.joblib


fold 1: macro F1 0.5235  (159,895 features) -> tfidf_logreg/fold1.joblib


fold 2: macro F1 0.5272  (160,415 features) -> tfidf_logreg/fold2.joblib


fold 3: macro F1 0.5236  (159,912 features) -> tfidf_logreg/fold3.joblib


fold 4: macro F1 0.5189  (159,523 features) -> tfidf_logreg/fold4.joblib
OOF macro F1 0.5222


## 2. Compare with the saved predictions used by the main notebook

In [4]:
so, st = np.load(OUT_DIR / "oof_tfidf.npy"), np.load(OUT_DIR / "test_tfidf.npy")
print(f"saved OOF macro F1 {mf1(y, so.argmax(1)):.4f} | this run {mf1(y, oof.argmax(1)):.4f}")
print(f"OOF argmax agreement {(so.argmax(1) == oof.argmax(1)).mean():.2%} | test argmax agreement {(st.argmax(1) == test_p.argmax(1)).mean():.2%}")
print(f"max |difference| in probabilities: OOF {np.abs(so - oof).max():.4f}, test {np.abs(st - test_p).max():.4f}")

saved OOF macro F1 0.5219 | this run 0.5222
OOF argmax agreement 99.17% | test argmax agreement 99.61%
max |difference| in probabilities: OOF 0.0564, test 0.0257


## 3. Predict new posts with a saved fold model

In [5]:
bundle = joblib.load(MODEL_DIR / "fold0.joblib")
posts = ["what a wonderful day", "you people are the worst"]            # synthetic examples
X = hstack([bundle["word_vectorizer"].transform(posts), bundle["char_vectorizer"].transform(posts)]).tocsr()
pd.DataFrame(bundle["logreg"].predict_proba(X), index=posts, columns=["p(0 explicit)", "p(1 subtle)", "p(2 neutral)"]).round(3)

,p(0 explicit),p(1 subtle),p(2 neutral)
what a wonderful day,0.187,0.463,0.350
you people are the worst,0.922,0.075,0.003
